# Feature Engineering: Transformation, Selection & Leakage Prevention Lab

Feature engineering bridges raw real-world data and machine learning algorithms. High-capacity models fail when presented with unscaled outliers, high-cardinality leakage, or ill-conditioned distributions.

### Table of Contents
1. [Missing Value Imputation & Indicator Mechanics](#1-missing-value-imputation--indicator-mechanics)
2. [Categorical Encoding: One-Hot vs. Smoothed Target Encoding](#2-categorical-encoding-one-hot-vs-smoothed-target-encoding)
3. [Numerical Transformations: Standard, Robust & Power Transforms](#3-numerical-transformations-standard-robust--power-transforms)
4. [Feature Selection: Filter, Wrapper, and Embedded Methods](#4-feature-selection-filter-wrapper-and-embedded-methods)
5. [Data Leakage Demonstration & Pipeline Integrity](#5-data-leakage-demonstration--pipeline-integrity)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

sns.set_theme(style="whitegrid")
print("Loaded Feature Engineering interactive lab environment.")

## 1. Missing Value Imputation & Indicator Mechanics
When data is Missing Not at Random (MNAR), missingness itself is informative. Appending a binary missingness indicator column preserves this predictive signal.

In [ ]:
np.random.seed(42)
income = np.random.lognormal(mean=10.5, sigma=0.8, size=1000)
# High-earners frequently decline to report income (MNAR)
missing_prob = np.clip((income - np.median(income)) / np.std(income) * 0.3, 0.05, 0.6)
is_missing = np.random.binomial(1, missing_prob).astype(bool)
income_observed = income.copy()
income_observed[is_missing] = np.nan

print(f"Total Samples: {len(income)} | Missing Count: {np.sum(is_missing)} ({np.mean(is_missing):.1%})")
print(f"Mean True Income:           ${np.mean(income):,.2f}")
print(f"Mean Observed (Excl. NaNs): ${np.nanmean(income_observed):,.2f}")

## 2. Categorical Encoding: One-Hot vs. Smoothed Target Encoding
One-hot encoding high-cardinality features (e.g., Zip Codes) causes extreme dimensional explosion and sparsity. Target encoding replaces each category with the expected target value smoothed with empirical Bayes priors:
$$S_c = \frac{n_c}{n_c + m} \bar{y}_c + \frac{m}{n_c + m} \bar{y}_{\text{global}}$$

In [ ]:
# High cardinality categorical feature (100 categories)
categories = [f"cat_{i}" for i in np.random.randint(0, 100, size=1000)]
targets = np.random.binomial(1, 0.25, size=1000)

# Compute smoothed target encoding
df_cat = pd.DataFrame({"category": categories, "target": targets})
global_mean = df_cat["target"].mean()
smoothing = 10.0

stats = df_cat.groupby("category")["target"].agg(["count", "mean"])
stats["smooth"] = (stats["count"] * stats["mean"] + smoothing * global_mean) / (stats["count"] + smoothing)

print("Sample of Smoothed Target Encoded Values:")
print(stats.head(6))

## 3. Numerical Transformations: Standard, Robust & Power Transforms
Visualizing how Yeo-Johnson power transformations normalize heavily skewed log-normal features.

In [ ]:
np.random.seed(0)
raw_skewed = np.random.exponential(scale=3.0, size=(1000, 1))

scaler_std = StandardScaler().fit_transform(raw_skewed)
scaler_rob = RobustScaler().fit_transform(raw_skewed)
transformer_yj = PowerTransformer(method="yeo-johnson").fit_transform(raw_skewed)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
axes[0].hist(raw_skewed, bins=30, color="#1f77b4")
axes[0].set_title("Raw Skewed Distribution", fontweight="bold")

axes[1].hist(scaler_std, bins=30, color="#ff7f0e")
axes[1].set_title("StandardScaler (Mean=0, Std=1)", fontweight="bold")

axes[2].hist(scaler_rob, bins=30, color="#2ca02c")
axes[2].set_title("RobustScaler (Median, IQR)", fontweight="bold")

axes[3].hist(transformer_yj, bins=30, color="#d62728")
axes[3].set_title("Yeo-Johnson (Gaussianized)", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Feature Selection: Filter, Wrapper & Embedded Methods
Ranking feature importance via Mutual Information (capturing non-linear dependencies) vs Recursive Feature Elimination (RFE).

In [ ]:
X_sel, y_sel = make_classification(n_samples=500, n_features=12, n_informative=4, random_state=42)
mi_scores = mutual_info_classif(X_sel, y_sel, random_state=42)

rfe = RFE(estimator=LogisticRegression(), n_features_to_select=4).fit(X_sel, y_sel)

plt.figure(figsize=(9, 4))
plt.bar(range(12), mi_scores, color="#2b5c8f")
plt.xticks(range(12), [f"F{i}" for i in range(12)])
plt.title("Mutual Information Feature Importance (Top Informative Features Identified)", fontweight="bold")
plt.xlabel("Feature")
plt.ylabel("Mutual Information Score")
plt.tight_layout()
plt.show()
print(f"RFE Selected Feature Indices: {np.where(rfe.support_)[0]}")

## 5. Data Leakage Demonstration & Pipeline Integrity
Demonstrating the subtle disaster of **Pre-Split Transformation**: Scaling or selecting features on the full dataset before cross-validation leaks test variance and leads to overly optimistic evaluation metrics.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline

# Synthetic completely random noise data: Ground truth accuracy should be exactly 50%
np.random.seed(42)
X_noise = np.random.randn(200, 2000)
y_noise = np.random.randint(0, 2, size=200)

# BAD LEAKAGE PIPELINE: Select top 20 features before CV split
mi_all = mutual_info_classif(X_noise, y_noise, random_state=42)
top_20 = np.argsort(mi_all)[-20:]
X_leaked = X_noise[:, top_20]
leaked_score = np.mean(cross_val_score(LogisticRegression(), X_leaked, y_noise, cv=5))

# PROPER PIPELINE: Feature selection strictly inside each CV fold
pipe = Pipeline([
    ("select", RFE(LogisticRegression(), n_features_to_select=20)),
    ("clf", LogisticRegression())
])
proper_score = np.mean(cross_val_score(pipe, X_noise, y_noise, cv=5))

print(f"Leaked CV Accuracy on PURE NOISE:  {leaked_score:.2%} (Spurious overconfidence!)")
print(f"Proper Un-Leaked CV Accuracy:      {proper_score:.2%} (True random baseline ~50%)")